# Drug Target Agent — Full 4-Stage Pipeline Test

Single L-R pair (TGFB1 → TGFBR2)에 대해 Drug Target Agent의 12개 @tool을 순서대로 테스트합니다.

| Stage | Tools | Description |
|-------|-------|-------------|
| 1 | #1-#5 | Biological Context (Census, KEGG, Reactome, STRING) |
| 2 | #6-#8 | Target Validation (OpenTargets, GWAS) |
| 3 | #9-#10 | Deep Dive (DrugBank, PubMed) |
| 4 | #11 | Structural Druggability (AlphaFold) |

## Setup

In [1]:
import json
import time
import os
import sys

# 프로젝트 루트를 path에 추가
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/jungj2/Dropbox/Metadata/aiagent/DeepMAST


In [2]:
from src.agents.drug_target.tools import (
    # Stage 1: Biological Context
    query_cellxgene_lr_context,       # #1
    query_kegg_pathways,              # #2
    query_reactome_pathways,          # #3
    query_string_ppi_pair,            # #4
    run_compare_census_lr_pair,       # #5
    # Stage 2: Target Validation
    query_opentargets_target,         # #6
    query_opentargets_known_drugs,    # #7
    query_gwas_gene_trait,            # #8
    # Stage 3: Deep Dive
    query_drugbank_target_profile,    # #9
    query_pubmed_clinical_limitations,# #10
    # Stage 4: Structural Druggability
    query_alphafold_plddt,            # #11
)

print("All 11 domain tools imported OK")

All 11 domain tools imported OK


In [3]:
# 테스트 파라미터
DISEASE = "breast cancer"
LIGAND = "TGFB1"
RECEPTOR = "TGFBR2"
TARGET_CELL_TYPE = "macrophage"
ORGANISM = "homo_sapiens"
DATA_DIR = "check_drugtarget"

# UniProt IDs for AlphaFold
UNIPROT_LIGAND = "P01137"   # TGFB1
UNIPROT_RECEPTOR = "P37173" # TGFBR2

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "evidence"), exist_ok=True)

print(f"Disease: {DISEASE}")
print(f"L-R pair: {LIGAND} -> {RECEPTOR}")
print(f"Target cell type: {TARGET_CELL_TYPE}")
print(f"Data dir: {DATA_DIR}")

Disease: breast cancer
L-R pair: TGFB1 -> TGFBR2
Target cell type: macrophage
Data dir: check_drugtarget


In [4]:
# Helper: run tool and collect timing
timings = {}
results = {}

def run_tool(name, tool_fn, params):
    """Run a tool, parse JSON result, store timing."""
    print(f"\n{'='*60}")
    print(f"Running: {name}")
    print(f"Params: {params}")
    print(f"{'='*60}")
    t0 = time.time()
    try:
        raw = tool_fn.invoke(params)
        elapsed = time.time() - t0
        result = json.loads(raw)
        timings[name] = elapsed
        results[name] = result
        status = "OK" if result.get("executed", False) else "FAIL"
        errors = result.get("errors", [])
        print(f"Status: {status} | Time: {elapsed:.1f}s")
        if errors:
            print(f"Errors: {errors}")
        return result
    except Exception as e:
        elapsed = time.time() - t0
        timings[name] = elapsed
        results[name] = {"executed": False, "errors": [str(e)]}
        print(f"EXCEPTION: {e} | Time: {elapsed:.1f}s")
        return results[name]

print("Helper ready")

Helper ready


---
## Stage 1: Biological Context (Tools #1-#5)

In [5]:
# #1 CELLxGENE Census — L-R pair 세포타입별 발현
r1 = run_tool("#1 cellxgene_lr_context", query_cellxgene_lr_context, {
    "ligand": LIGAND,
    "receptor": RECEPTOR,
    "disease": DISEASE,
    "organism": ORGANISM,
})
print(f"\nCell count: {r1.get('cell_count', 0):,}")
print(f"Available genes: {r1.get('available_genes', [])}")
for ct in r1.get('top_cell_types', []):
    print(f"  {ct['cell_type']}: {ct['count']:,}")


Running: #1 cellxgene_lr_context
Params: {'ligand': 'TGFB1', 'receptor': 'TGFBR2', 'disease': 'breast cancer', 'organism': 'homo_sapiens'}


The "stable" release is currently 2025-11-08. Specify 'census_version="2025-11-08"' in future calls to open_soma() to ensure data consistency.


Status: OK | Time: 5.7s

Cell count: 1,101,430
Available genes: ['TGFB1', 'TGFBR2']
  malignant cell: 576,171
  macrophage: 80,485
  T cell: 75,981
  hepatocyte: 67,435
  fibroblast: 62,959


In [9]:
# #2 KEGG Pathways — Ligand
r2a = run_tool("#2a kegg_ligand", query_kegg_pathways, {
    "gene_symbol": LIGAND,
})
for pw in r2a.get('pathways', []):
    print(f"  {pw['pathway_id']}: {pw['name']}")


Running: #2a kegg_ligand
Params: {'gene_symbol': 'TGFB1'}
Status: OK | Time: 2.0s


In [7]:
# #2 KEGG Pathways — Receptor
r2b = run_tool("#2b kegg_receptor", query_kegg_pathways, {
    "gene_symbol": RECEPTOR,
})
for pw in r2b.get('pathways', []):
    print(f"  {pw['pathway_id']}: {pw['name']}")


Running: #2b kegg_receptor
Params: {'gene_symbol': 'TGFBR2'}
Status: OK | Time: 6.4s
  hsa04010: MAPK signaling pathway - Homo sapiens (human)
  hsa04060: Cytokine-cytokine receptor interaction - Homo sapiens (human)
  hsa04068: FoxO signaling pathway - Homo sapiens (human)
  hsa04144: Endocytosis - Homo sapiens (human)
  hsa04218: Cellular senescence - Homo sapiens (human)


In [10]:
# #3 Reactome Pathways — Ligand
r3a = run_tool("#3a reactome_ligand", query_reactome_pathways, {
    "gene_symbol": LIGAND,
})
for pw in r3a.get('pathways', []):
    print(f"  {pw['reactome_id']}: {pw['name']}")


Running: #3a reactome_ligand
Params: {'gene_symbol': 'TGFB1'}
Status: OK | Time: 0.4s
  R-HSA-2173789: TGF-beta receptor signaling activates SMADs
  R-HSA-3642279: TGFBR2 MSI Frameshift Mutants in Cancer
  R-HSA-3645790: TGFBR2 Kinase Domain Mutants in Cancer
  R-HSA-8941855: RUNX3 regulates CDKN1A transcription
  R-HSA-9762293: Regulation of CDH11 gene transcription


In [11]:
# #3 Reactome Pathways — Receptor
r3b = run_tool("#3b reactome_receptor", query_reactome_pathways, {
    "gene_symbol": RECEPTOR,
})
for pw in r3b.get('pathways', []):
    print(f"  {pw['reactome_id']}: {pw['name']}")


Running: #3b reactome_receptor
Params: {'gene_symbol': 'TGFBR2'}
Status: OK | Time: 0.3s
  R-HSA-3642278: Loss of Function of TGFBR2 in Cancer
  R-HSA-3642279: TGFBR2 MSI Frameshift Mutants in Cancer
  R-HSA-3645790: TGFBR2 Kinase Domain Mutants in Cancer
  R-HSA-2173789: TGF-beta receptor signaling activates SMADs
  R-HSA-3304351: Signaling by TGF-beta Receptor Complex in Cancer


In [12]:
# #4 STRING PPI
r4 = run_tool("#4 string_ppi", query_string_ppi_pair, {
    "gene_a": LIGAND,
    "gene_b": RECEPTOR,
})
print(f"\nInteraction found: {r4.get('interaction_found')}")
print(f"Max combined score: {r4.get('max_combined_score')}")
if r4.get('ppi_enrichment'):
    print(f"PPI enrichment: {r4['ppi_enrichment']}")


Running: #4 string_ppi
Params: {'gene_a': 'TGFB1', 'gene_b': 'TGFBR2'}
Status: OK | Time: 0.5s

Interaction found: True
Max combined score: 0.999
PPI enrichment: {'number_of_nodes': 2, 'number_of_edges': 1, 'average_node_degree': 1.0, 'local_clustering_coefficient': 1.0, 'expected_number_of_edges': 0, 'p_value': 0.1352}


In [13]:
# #5 Census Compare (z-test) — 시간이 오래 걸림 (~90s)
r5 = run_tool("#5 census_compare", run_compare_census_lr_pair, {
    "data_dir": DATA_DIR,
    "disease": DISEASE,
    "gene_a": LIGAND,
    "gene_b": RECEPTOR,
    "target_cell_type": TARGET_CELL_TYPE,
    "organism": ORGANISM,
    "min_cells": 50,
    "top_n": 20,
    "timeout_sec": 180,
})
if r5.get('target_vs_rest'):
    tvr = r5['target_vs_rest']
    print(f"\nTarget vs Rest ({TARGET_CELL_TYPE}):")
    for gene_key in ['gene_a', 'gene_b', 'coexpression']:
        if gene_key in tvr:
            d = tvr[gene_key]
            print(f"  {gene_key}: target={d.get('target_pct',0):.1%} rest={d.get('rest_pct',0):.1%} RR={d.get('risk_ratio',0):.2f} p={d.get('p_value','?')}")


Running: #5 census_compare
Params: {'data_dir': 'check_drugtarget', 'disease': 'breast cancer', 'gene_a': 'TGFB1', 'gene_b': 'TGFBR2', 'target_cell_type': 'macrophage', 'organism': 'homo_sapiens', 'min_cells': 50, 'top_n': 20, 'timeout_sec': 180}
Status: OK | Time: 103.0s

Target vs Rest (macrophage):
  gene_a: target=23.5% rest=14.1% RR=1.66 p=1e-323
  gene_b: target=26.8% rest=15.1% RR=1.78 p=1e-323
  coexpression: target=10.0% rest=4.1% RR=2.43 p=1e-323


---
## Stage 2: Target Validation & Scoring (Tools #6-#8)

In [15]:
# #6 OpenTargets Target — Ligand
r6a = run_tool("#6a opentargets_ligand", query_opentargets_target, {
    "gene_symbol": LIGAND,
    "disease": DISEASE,
})
print(f"\nEnsembl ID: {r6a.get('ensembl_id')}")
print(f"EFO ID: {r6a.get('efo_id')}")
ti = r6a.get('target_info', {})
if ti:
    print(f"Tractability: {ti.get('tractability', 'N/A')}")
    print(f"Safety: {ti.get('safety', 'N/A')}")
print(f"Disease evidence count: {len(r6a.get('disease_evidence', []))}")
print(f"Associations count: {len(r6a.get('associations', []))}")


Running: #6a opentargets_ligand
Params: {'gene_symbol': 'TGFB1', 'disease': 'breast cancer'}
Status: FAIL | Time: 0.4s
Errors: ['OpenTargets query failed: API request failed: 400 Client Error: Bad Request for url: https://api.platform.opentargets.org/api/v4/graphql']

Ensembl ID: None
EFO ID: None
Disease evidence count: 0
Associations count: 0


In [ ]:
# #6 OpenTargets Target — Receptor
r6b = run_tool("#6b opentargets_receptor", query_opentargets_target, {
    "gene_symbol": RECEPTOR,
    "disease": DISEASE,
})
print(f"\nEnsembl ID: {r6b.get('ensembl_id')}")
print(f"EFO ID: {r6b.get('efo_id')}")
ti = r6b.get('target_info', {})
if ti:
    print(f"Tractability: {ti.get('tractability', 'N/A')}")
    print(f"Safety: {ti.get('safety', 'N/A')}")
print(f"Disease evidence count: {len(r6b.get('disease_evidence', []))}")
print(f"Associations count: {len(r6b.get('associations', []))}")

In [ ]:
# #7 OpenTargets Known Drugs
r7 = run_tool("#7 opentargets_drugs", query_opentargets_known_drugs, {
    "disease": DISEASE,
})
print(f"\nEFO ID: {r7.get('efo_id')}")
drugs = r7.get('drugs', {})
if isinstance(drugs, dict):
    print(f"Drug count: {drugs.get('count', len(drugs))}")
elif isinstance(drugs, list):
    print(f"Drug count: {len(drugs)}")
    for d in drugs[:5]:
        print(f"  {d}")

In [ ]:
# #8 GWAS — Ligand
r8a = run_tool("#8a gwas_ligand", query_gwas_gene_trait, {
    "gene_symbol": LIGAND,
    "disease": DISEASE,
})
print(f"\nEFO traits: {len(r8a.get('efo_traits', []))}")
print(f"Trait associations: {len(r8a.get('trait_associations', []))}")
print(f"Gene associations: {len(r8a.get('gene_associations', []))}")

In [ ]:
# #8 GWAS — Receptor
r8b = run_tool("#8b gwas_receptor", query_gwas_gene_trait, {
    "gene_symbol": RECEPTOR,
    "disease": DISEASE,
})
print(f"\nEFO traits: {len(r8b.get('efo_traits', []))}")
print(f"Trait associations: {len(r8b.get('trait_associations', []))}")
print(f"Gene associations: {len(r8b.get('gene_associations', []))}")

---
## Stage 3: Deep Dive (Tools #9-#10)

In [ ]:
# #9 DrugBank — Ligand
r9a = run_tool("#9a drugbank_ligand", query_drugbank_target_profile, {
    "gene_symbol": LIGAND,
})
print(f"\nSource: {r9a.get('source')}")
print(f"Matched drugs: {len(r9a.get('matched_drugs', []))}")
for d in r9a.get('matched_drugs', [])[:5]:
    print(f"  {d.get('drug_name', '?')} ({d.get('drugbank_id') or d.get('chembl_id', '?')})")

In [ ]:
# #9 DrugBank — Receptor
r9b = run_tool("#9b drugbank_receptor", query_drugbank_target_profile, {
    "gene_symbol": RECEPTOR,
})
print(f"\nSource: {r9b.get('source')}")
print(f"Matched drugs: {len(r9b.get('matched_drugs', []))}")
for d in r9b.get('matched_drugs', [])[:5]:
    print(f"  {d.get('drug_name', '?')} ({d.get('drugbank_id') or d.get('chembl_id', '?')})")

In [ ]:
# #10 PubMed Clinical Limitations — both genes
r10a = run_tool("#10a pubmed_ligand", query_pubmed_clinical_limitations, {
    "target_name": LIGAND,
})
print(f"\nPapers found: {len(r10a.get('papers', []))}")
for p in r10a.get('papers', []):
    print(f"  [{p.get('year', '?')}] PMID:{p.get('pmid', '?')} - {p.get('title', '?')[:80]}")

In [ ]:
r10b = run_tool("#10b pubmed_receptor", query_pubmed_clinical_limitations, {
    "target_name": RECEPTOR,
})
print(f"\nPapers found: {len(r10b.get('papers', []))}")
for p in r10b.get('papers', []):
    print(f"  [{p.get('year', '?')}] PMID:{p.get('pmid', '?')} - {p.get('title', '?')[:80]}")

---
## Stage 4: Structural Druggability (Tool #11)

In [ ]:
# #11 AlphaFold pLDDT — Ligand
r11a = run_tool("#11a alphafold_ligand", query_alphafold_plddt, {
    "uniprot_id": UNIPROT_LIGAND,
})
print(f"\nEntry ID: {r11a.get('entry_id')}")
print(f"Mean pLDDT: {r11a.get('mean_plddt')}")

In [ ]:
# #11 AlphaFold pLDDT — Receptor
r11b = run_tool("#11b alphafold_receptor", query_alphafold_plddt, {
    "uniprot_id": UNIPROT_RECEPTOR,
})
print(f"\nEntry ID: {r11b.get('entry_id')}")
print(f"Mean pLDDT: {r11b.get('mean_plddt')}")

---
## Summary

In [ ]:
# Timing summary
print(f"{'Tool':<30} {'Time (s)':>10} {'Status':>8}")
print("-" * 50)
total = 0
for name, t in timings.items():
    status = "OK" if results.get(name, {}).get('executed', False) else "FAIL"
    print(f"{name:<30} {t:>10.1f} {status:>8}")
    total += t
print("-" * 50)
print(f"{'TOTAL':<30} {total:>10.1f}")
print(f"\nSuccess: {sum(1 for r in results.values() if r.get('executed'))}/{len(results)}")

In [ ]:
# Bar chart
import matplotlib.pyplot as plt

names = list(timings.keys())
times = list(timings.values())
colors = ['green' if results.get(n, {}).get('executed') else 'red' for n in names]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(names, times, color=colors, alpha=0.8)
ax.set_xlabel('Time (seconds)')
ax.set_title(f'Drug Target Tool Timing — {LIGAND}→{RECEPTOR} ({DISEASE})')
for bar, t in zip(bars, times):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{t:.1f}s', va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Save all results to JSON
output_path = os.path.join(DATA_DIR, "test_all_stages.json")
with open(output_path, "w") as f:
    json.dump({
        "parameters": {
            "disease": DISEASE,
            "ligand": LIGAND,
            "receptor": RECEPTOR,
            "target_cell_type": TARGET_CELL_TYPE,
            "organism": ORGANISM,
            "uniprot_ligand": UNIPROT_LIGAND,
            "uniprot_receptor": UNIPROT_RECEPTOR,
        },
        "timings": timings,
        "results": results,
    }, f, indent=2, default=str)

print(f"Saved: {output_path}")